> **Provenance.** Written during the author's research internship at the National Institute of Statistics (NIS), Cambodia, using Cambodia Agriculture Survey (CAS) 2024 holding-level survey data. **The survey data are confidential and are NOT included in this repository**, so this notebook cannot be run here as-is. All cell outputs have been removed. File paths were rewritten from private local paths to relative `data/...` paths during packaging; the analysis code is otherwise unchanged. See `data/README.md` for the data access statement.
>
> **This notebook.** Parcel-level rice **yield** prediction (LightGBM, XGBoost, Random Forest, Gradient Boosting, voting ensemble). **Validity warning:** an independent review at packaging time found target leakage in the feature `storage_ratio` (its denominator contains the target `yield_total(kg/ha)`) and a random parcel-level split. The recorded accuracy is therefore an optimistic upper bound. See `reports/validity_review.md`. Original filename: `v4.ipynb`.


In [ ]:
# ============================================
# ULTIMATE Rice Yield Prediction - OPTIMIZED VERSION
# ============================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PowerTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.impute import SimpleImputer
from sklearn.ensemble import VotingRegressor, StackingRegressor
from sklearn.linear_model import ElasticNet
import optuna

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt


In [ ]:
# ==============================
# 1. Load Data
# ==============================
df = pd.read_csv("final_dataset.csv")

print(f"Dataset shape: {df.shape}")
print(f"Target variable range: {df['yield_total(kg/ha)'].min():.2f} - {df['yield_total(kg/ha)'].max():.2f}")

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
import seaborn as sns

# ==============================
# DATA CORRELATION ANALYSIS
# ==============================

# Select numeric columns only
numeric_features = df.select_dtypes(include=['float64', 'int64']).columns
correlations = df[numeric_features].corr()['yield_total(kg/ha)'].sort_values(ascending=False)

print("Top 15 Positive Correlations with Yield:")
print(correlations[1:16])  # Skip first (correlation with itself)
print("\nTop 15 Negative Correlations with Yield:")
print(correlations[-15:])

# Create correlation matrix visualization
import matplotlib.pyplot as plt

# Select top correlated features
top_features = correlations.abs().nlargest(15).index
correlation_matrix = df[top_features].corr()

plt.figure(figsize=(12, 8))
sns.heatmap(correlation_matrix, 
			annot=True, 
			cmap='coolwarm', 
			center=0, 
			fmt='.2f',
			square=True)
plt.title('Correlation Matrix of Top Features with Yield')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8')  # Use a clean seaborn style

# List of variables and titles
plots = {
    'yield_total(kg/ha)': 'Yield Distribution (kg/ha)',
    'area_harvested_ha': 'Harvested Area Distribution (ha)',
    'production_total(kg)': 'Production Distribution (kg)',
    'stored_qty_total(kg)': 'Storage Distribution (kg)'
}

# Create individual plots
for col, title in plots.items():
    plt.figure(figsize=(8, 6))
    ax = sns.boxplot(y=df[col])
    ax2 = ax.twinx()
    sns.kdeplot(data=df[col], ax=ax2, color='blue')
    ax.set_title(title, fontsize=14)
    ax.set_ylabel(col)
    plt.show()

# Print summary statistics
for col in plots.keys():
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)][col].count()
    
    print(f"\n{col}:")
    print(f"IQR range: {lower_bound:.2f} - {upper_bound:.2f}")
    print(f"Actual range: {df[col].min():.2f} - {df[col].max():.2f}")
    print(f"Number of outliers: {outliers} ({(outliers/len(df)*100):.2f}%)")


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8')  # Style

# Variables and titles
plots = {
    'yield_total(kg/ha)': 'Yield Distribution (kg/ha)',
    'area_harvested_ha': 'Harvested Area Distribution (ha)',
    'production_total(kg)': 'Production Distribution (kg)',
    'stored_qty_total(kg)': 'Storage Distribution (kg)'
}

# Create individual plots with histogram + KDE
for col, title in plots.items():
    plt.figure(figsize=(8, 6))
    sns.histplot(df[col], kde=True, bins=30, color="skyblue")
    plt.title(title, fontsize=14)
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.show()

# Print summary statistics
for col in plots.keys():
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)][col].count()
    
    print(f"\n{col}:")
    print(f"IQR range: {lower_bound:.2f} - {upper_bound:.2f}")
    print(f"Actual range: {df[col].min():.2f} - {df[col].max():.2f}")
    print(f"Number of outliers: {outliers} ({(outliers/len(df)*100):.2f}%)")


In [ ]:
# ==============================
# 2. Enhanced Data Cleaning & Outlier Handling
# ==============================

# Drop leakage features and identifiers
leakage_features = ["production_total(kg)", "int_key", "int_id", "parcel_id", "crop_field"]
existing_leakage_features = [col for col in leakage_features if col in df.columns]
df = df.drop(columns=existing_leakage_features)
print(f"Dropped leakage features: {existing_leakage_features}")

# More aggressive outlier handling for yield
original_count = len(df)
Q1 = df['yield_total(kg/ha)'].quantile(0.01)  # More conservative
Q3 = df['yield_total(kg/ha)'].quantile(0.99)
IQR = Q3 - Q1
lower_bound = max(Q1 - 1.5 * IQR, 300)  # Minimum realistic yield for Cambodia
upper_bound = min(Q3 + 1.5 * IQR, 6000) # Maximum realistic yield for Cambodia

df_clean = df[(df['yield_total(kg/ha)'] >= lower_bound) & (df['yield_total(kg/ha)'] <= upper_bound)].copy()
print(f"Removed {original_count - len(df_clean)} outliers from target variable")
print(f"New target range: {df_clean['yield_total(kg/ha)'].min():.2f} - {df_clean['yield_total(kg/ha)'].max():.2f}")

# Remove outliers from important features
outlier_features = ['area_planted_ha', 'area_harvested_ha', 'stored_qty_total(kg)']
for feature in outlier_features:
    if feature in df_clean.columns:
        Q1_f = df_clean[feature].quantile(0.01)
        Q3_f = df_clean[feature].quantile(0.99)
        IQR_f = Q3_f - Q1_f
        lower_f = Q1_f - 3 * IQR_f
        upper_f = Q3_f + 3 * IQR_f
        before = len(df_clean)
        df_clean = df_clean[(df_clean[feature] >= lower_f) & (df_clean[feature] <= upper_f)]
        print(f"Removed {before - len(df_clean)} outliers from {feature}")

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import LabelEncoder  # For potential categorical encoding

### Feature Engineering


In [ ]:
# ==============================

# Drop province if exists, keep zone
if "prov" in df_clean.columns:
    df_clean = df_clean.drop(columns=["prov"])

# 3.1 Enhanced Harvest efficiency features
df_clean["harvest_ratio"] = df_clean["area_harvested_ha"] / (df_clean["area_planted_ha"] + 1e-6)
df_clean["loss_area"] = df_clean["area_planted_ha"] - df_clean["area_harvested_ha"]
df_clean["loss_ratio"] = df_clean["loss_area"] / (df_clean["area_planted_ha"] + 1e-6)
df_clean["harvest_efficiency"] = np.log1p(df_clean["harvest_ratio"] * 100)  # Log transform

# 3.2 Enhanced Stress and problem indicators
stress_cols = ["reason_drought","reason_flood","reason_nat_event",
               "reason_pest","reason_theft","reason_sickness",
               "reason_no_labor","reason_other"]
existing_stress_cols = [col for col in stress_cols if col in df_clean.columns]
df_clean["stress_count"] = df_clean[existing_stress_cols].sum(axis=1)
df_clean["has_stress"] = (df_clean["stress_count"] > 0).astype(int)

# 3.3 Advanced Resource utilization features
df_clean["fert_irrig"] = df_clean["fert_applied"] * df_clean["irrigated"]
df_clean["pest_irrig"] = df_clean["pest_applied"] * df_clean["irrigated"]
df_clean["resource_intensity"] = df_clean["fert_applied"] + df_clean["pest_applied"] + df_clean["irrigated"]
df_clean["management_score"] = (df_clean["fert_applied"] * 0.4 + 
                               df_clean["pest_applied"] * 0.3 + 
                               df_clean["irrigated"] * 0.3)

# 3.4 Enhanced Crop grouping with interactions
df_clean["crop_group"] = df_clean["crop_code"].replace({101:"NonAromatic",102:"Aromatic",103:"Sticky"})
# Create crop-specific features
crop_dummies = pd.get_dummies(df_clean["crop_group"], prefix="crop")
df_clean = pd.concat([df_clean, crop_dummies], axis=1)

# 3.5 Advanced Production efficiency features
if "stored_qty_total(kg)" in df_clean.columns and "area_harvested_ha" in df_clean.columns:
    df_clean["storage_per_ha"] = df_clean["stored_qty_total(kg)"] / (df_clean["area_harvested_ha"] + 1e-6)
    df_clean["storage_ratio"] = df_clean["stored_qty_total(kg)"] / (df_clean["area_harvested_ha"] * df_clean['yield_total(kg/ha)'] + 1e-6)

# 3.6 Advanced Land productivity features
df_clean["land_productivity"] = df_clean["area_harvested_ha"] / (df_clean["area_planted_ha"] + 1e-6)
df_clean["area_efficiency"] = np.log1p(df_clean["area_harvested_ha"]) / (1 + df_clean["area_planted_ha"])

# Initialize categorical columns list before using it
categorical_cols = ["zone", "crop_group", "fert_type_org", "fert_type_inorg"]

# 3.7 GPS-based features (clustering and regional patterns)
if all(col in df_clean.columns for col in ["GPS_NEW__Latitude", "GPS_NEW__Longitude"]):
    from sklearn.cluster import KMeans
    gps_data = df_clean[["GPS_NEW__Latitude", "GPS_NEW__Longitude"]].dropna()
    if len(gps_data) > 10:
        kmeans = KMeans(n_clusters=8, random_state=42, n_init=10)
        df_clean["gps_cluster"] = kmeans.fit_predict(df_clean[["GPS_NEW__Latitude", "GPS_NEW__Longitude"]].fillna(method='ffill'))
        if "gps_cluster" not in categorical_cols:
            categorical_cols.append("gps_cluster")

# 3.8 Polynomial features for key variables
key_features = ['area_planted_ha', 'harvest_ratio', 'storage_per_ha']
for feature in key_features:
    if feature in df_clean.columns:
        df_clean[f"{feature}_squared"] = df_clean[feature] ** 2
        df_clean[f"{feature}_log"] = np.log1p(df_clean[feature])

# 3.9 Interaction features between top important features
if all(col in df_clean.columns for col in ['area_planted_ha', 'harvest_ratio']):
    df_clean["area_harvest_interaction"] = df_clean["area_planted_ha"] * df_clean["harvest_ratio"]
if all(col in df_clean.columns for col in ['storage_per_ha', 'management_score']):
    df_clean["storage_management_interaction"] = df_clean["storage_per_ha"] * df_clean["management_score"]

print(f"Total features after ULTIMATE engineering: {len(df_clean.columns)}")


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Get numeric columns
numeric_features = df_clean.select_dtypes(include=['float64', 'int64']).columns

# Calculate correlations with target variable
correlations = df_clean[numeric_features].corr()['yield_total(kg/ha)'].sort_values(ascending=False)

# === 1. Bar plot of top correlations ===
plt.figure(figsize=(10, 6))
correlations[1:16].plot(kind='bar', color="skyblue")
plt.title('Top 15 Positive Correlations with Yield', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.ylabel('Correlation Coefficient')
plt.show()

# === 2. Bar plot of bottom correlations ===
plt.figure(figsize=(10, 6))
correlations[-10:].plot(kind='bar', color="salmon")
plt.title('Top 10 Negative Correlations with Yield', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.ylabel('Correlation Coefficient')
plt.show()

# === 3. Heatmap of top correlated features ===
plt.figure(figsize=(12, 8))
top_features = correlations.abs().nlargest(15).index
correlation_matrix = df_clean[top_features].corr()

sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap='coolwarm',
    center=0,
    fmt='.2f',
    square=True
)
plt.title('Correlation Matrix of Top 15 Features', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.show()

# Print summary
print("\nTop 10 Positive Correlations with Yield:")
print(correlations[1:11])

print("\nTop 10 Negative Correlations with Yield:")
print(correlations[-10:])


In [ ]:
# Select all numeric columns
numeric_cols_all = df_clean.select_dtypes(include="number").columns.tolist()

# Compute correlation with target
corr_with_yield = df_clean[numeric_cols_all].corr()["yield_total(kg/ha)"].drop("yield_total(kg/ha)")

# Top 5 features with highest absolute correlation
top5_features = corr_with_yield.abs().sort_values(ascending=False).head(5)
print("Top 5 Features Most Correlated with Yield:")
print(corr_with_yield[top5_features.index])


In [ ]:
# ==============================
# 4. Define Features & Target
# ==============================
target = "yield_total(kg/ha)"

y = df_clean[target]
X = df_clean.drop(columns=[target])

# ==============================
# Robust Enhanced Feature Selection
# ==============================

# Get numeric columns that actually exist in X
numeric_cols = [col for col in X.select_dtypes(include=[np.number]).columns if col in X.columns]

if numeric_cols:
    # 1️⃣ Remove constant features
    constant_features = [col for col in numeric_cols if X[col].nunique() <= 1]
    if constant_features:
        print(f"Dropping constant features: {constant_features}")
        X = X.drop(columns=constant_features)
        # Update numeric_cols after dropping
        numeric_cols = [col for col in numeric_cols if col not in constant_features]

    # 2️⃣ Remove highly correlated features
    if len(numeric_cols) > 1:  # Need at least 2 columns to compute correlation
        correlation_matrix = X[numeric_cols].corr().abs()
        upper_tri = correlation_matrix.where(np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool))
        to_drop = [column for column in upper_tri.columns if any(upper_tri[column] > 0.90)]
        if to_drop:
            print(f"Dropping highly correlated features: {to_drop}")
            X = X.drop(columns=to_drop)
            numeric_cols = [col for col in numeric_cols if col not in to_drop]


In [ ]:
# Define numerical and categorical columns
numerical_cols = [col for col in X.columns if X[col].dtype in ['int64', 'float64']]
categorical_cols = [col for col in X.columns if X[col].dtype == 'object']

# Advanced preprocessing pipelines
num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
    # Removed PowerTransformer as it was causing issues
])

cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_pipe, numerical_cols),
        ("cat", cat_pipe, categorical_cols)
    ]
)

In [ ]:
# ==============================
# 6. Train/Test Split
# ==============================
stratify_col = df_clean["zone"] if "zone" in df_clean.columns else None
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=stratify_col  # Smaller test size
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")


In [ ]:
# ==============================
# 7. HYPER-OPTIMIZED Models
# ==============================

# Optimized based on your previous results
models = {
    "LightGBM_Optimal": LGBMRegressor(
        n_estimators=500, 
        learning_rate=0.05, 
        max_depth=10,
        num_leaves=63, 
        subsample=0.8, 
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=0.1,
        random_state=42, 
        n_jobs=-1,
        verbose=-1
    ),
    "XGBoost_Optimal": XGBRegressor(
        n_estimators=400,
        learning_rate=0.05,
        max_depth=8,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1
    ),
    "RandomForest_Optimal": RandomForestRegressor(
        n_estimators=300,
        max_depth=20,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features='sqrt',
        bootstrap=True,
        random_state=42,
        n_jobs=-1
    ),
    "GradientBoosting_Optimal": GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        max_features='sqrt',
        random_state=42
    )
}

In [ ]:
# ==============================
# 8. ENSEMBLE and ADVANCED Training
# ==============================
results = []
feature_importances = {}

print("\n" + "="*80)
print("TRAINING OPTIMIZED MODELS")
print("="*80)

for name, model in models.items():
    print(f"\nTraining {name}...")
    
    try:
        pipe = Pipeline(steps=[("preprocessor", preprocessor),
                               ("model", model)])
        
        # Enhanced cross-validation
        cv_scores = cross_val_score(pipe, X_train, y_train, 
                                   cv=5, scoring='r2', n_jobs=1)  # More folds
        
        # Full training
        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_test)

        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        mae = mean_absolute_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)
        
        # Store feature importance
        if hasattr(pipe.named_steps['model'], 'feature_importances_'):
            preprocessor.fit(X_train)
            feature_names = numerical_cols.copy()
            if 'cat' in preprocessor.named_transformers_:
                cat_features = list(preprocessor.named_transformers_['cat']
                                  .named_steps['onehot'].get_feature_names_out(categorical_cols))
                feature_names.extend(cat_features)
            
            importances = pipe.named_steps['model'].feature_importances_
            feature_importances[name] = pd.DataFrame({
                'feature': feature_names,
                'importance': importances
            }).sort_values('importance', ascending=False)

        results.append([name, rmse, mae, r2, cv_scores.mean(), cv_scores.std()])
        print(f"{name} - R²: {r2:.4f}, CV R²: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
        
    except Exception as e:
        print(f"Error training {name}: {e}")
        continue


In [ ]:
# ==============================
# 8. ENSEMBLE and ADVANCED Training
# ==============================

results = []
feature_importances = {}

print("\n" + "="*80)
print("TRAINING OPTIMIZED MODELS")
print("="*80)

for name, model in models.items():
    print(f"\nTraining {name}...")

    try:
        # Pipeline: preprocessing + model
        pipe = Pipeline(steps=[("preprocessor", preprocessor),
                               ("model", model)])

        # Cross-validation (5 folds) for stable performance estimate
        cv_scores = cross_val_score(pipe, X_train, y_train, 
                                    cv=5, scoring='r2', n_jobs=-1)  # Parallel CV

        # Full model training
        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_test)

        # Metrics
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        mae = mean_absolute_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)

        # Feature importance extraction (if supported)
        if hasattr(pipe.named_steps['model'], 'feature_importances_'):
            # Fit preprocessor to get feature names
            preprocessor.fit(X_train)
            
            feature_names = numerical_cols.copy()
            if 'cat' in preprocessor.named_transformers_:
                cat_features = list(preprocessor.named_transformers_['cat']
                                    .named_steps['onehot'].get_feature_names_out(categorical_cols))
                feature_names.extend(cat_features)

            importances = pipe.named_steps['model'].feature_importances_
            feature_importances[name] = pd.DataFrame({
                'feature': feature_names,
                'importance': importances
            }).sort_values('importance', ascending=False)

        # Store results
        results.append([name, rmse, mae, r2, cv_scores.mean(), cv_scores.std()])
        print(f"{name} - R²: {r2:.4f}, CV R²: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

    except Exception as e:
        print(f"Error training {name}: {e}")
        continue

# Convert results to DataFrame for easy viewing
results_df = pd.DataFrame(results, columns=["Model", "RMSE", "MAE", "R2", "CV_R2_mean", "CV_R2_std"])
results_df = results_df.sort_values("R2", ascending=False)
print("\nFinal Model Performance Summary:\n")
print(results_df)


In [ ]:
# ==============================
# 9. CREATE ENSEMBLE MODEL
# ==============================
print("\n" + "="*80)
print("CREATING ENSEMBLE MODEL")
print("="*80)

# Create ensemble of top 3 models
if len(results) >= 3:
    results_df = pd.DataFrame(results, columns=["Model", "RMSE", "MAE", "R²", "CV R² Mean", "CV R² Std"])
    top_3_models = results_df.nlargest(3, 'R²')['Model'].tolist()
    
    print(f"Top 3 models for ensemble: {top_3_models}")
    
    # Create voting ensemble
    estimators = []
    for model_name in top_3_models:
        if model_name in models:
            pipe = Pipeline(steps=[("preprocessor", preprocessor),
                                   ("model", models[model_name])])
            estimators.append((model_name, pipe))
    
    if len(estimators) >= 2:
        # Voting Regressor
        voting_regressor = VotingRegressor(estimators=estimators)
        voting_regressor.fit(X_train, y_train)
        y_pred_voting = voting_regressor.predict(X_test)
        voting_r2 = r2_score(y_test, y_pred_voting)
        voting_rmse = np.sqrt(mean_squared_error(y_test, y_pred_voting))
        
        results.append(["Voting_Ensemble", voting_rmse, mean_absolute_error(y_test, y_pred_voting), 
                       voting_r2, np.nan, np.nan])
        print(f"Voting Ensemble - R²: {voting_r2:.4f}, RMSE: {voting_rmse:.2f}")


In [ ]:
# ==============================
# 10. FINAL RESULTS & ANALYSIS
# ==============================
if results:
    results_df = pd.DataFrame(results, columns=["Model", "RMSE", "MAE", "R²", "CV R² Mean", "CV R² Std"])
    print("\n" + "="*80)
    print("FINAL MODEL PERFORMANCE COMPARISON")
    print("="*80)
    print(results_df.sort_values(by="R²", ascending=False).round(4))

    # Select best model
    best_model_name = results_df.loc[results_df['R²'].idxmax(), 'Model']
    print(f"\n🏆 BEST MODEL: {best_model_name}")

    # Train final best model on full training data
    if best_model_name == "Voting_Ensemble":
        final_model = voting_regressor
    else:
        final_model = Pipeline(steps=[("preprocessor", preprocessor),
                                    ("model", models[best_model_name])])
        final_model.fit(X_train, y_train)

    # Final predictions
    y_pred_final = final_model.predict(X_test)
    
    # Enhanced evaluation metrics
    from sklearn.metrics import mean_absolute_percentage_error
    
    final_rmse = np.sqrt(mean_squared_error(y_test, y_pred_final))
    final_mae = mean_absolute_error(y_test, y_pred_final)
    final_r2 = r2_score(y_test, y_pred_final)
    mape = mean_absolute_percentage_error(y_test, y_pred_final)
    
    # Calculate within 10% accuracy
    within_10_percent = np.mean(np.abs((y_test - y_pred_final) / y_test) <= 0.1) * 100

    print(f"\n🎯 FINAL MODEL PERFORMANCE:")
    print(f"R²: {final_r2:.4f}")
    print(f"RMSE: {final_rmse:.2f}")
    print(f"MAE: {final_mae:.2f}")
    print(f"MAPE: {mape:.2%}")
    print(f"Within 10% accuracy: {within_10_percent:.1f}%")


In [ ]:
# ==============================
# 11. ADVANCED FEATURE ANALYSIS
# ==============================
print("\n" + "="*80)
print("ADVANCED FEATURE ANALYSIS")
print("="*80)

if best_model_name in feature_importances:
	important_features = feature_importances[best_model_name]
	
	print(f"\n📊 Top 15 Most Important Features:")
	top_15 = important_features.head(15)
	for idx, row in top_15.iterrows():
		# Calculate correlation for numeric features
		if row['feature'] in X.columns and X[row['feature']].dtype in ['int64', 'float64']:
			corr = np.corrcoef(X[row['feature']], y)[0,1]
			print(f"  {idx+1:2d}. {row['feature']}: {row['importance']:.1f} (corr: {corr:.3f})")
		else:
			print(f"  {idx+1:2d}. {row['feature']}: {row['importance']:.1f}")


In [ ]:
# ==============================
# 12. PREDICTION INSIGHTS
# ==============================
print("\n" + "="*80)
print("PREDICTION INSIGHTS")
print("="*80)

residuals = y_test - y_pred_final
residual_std = residuals.std()

print(f"\n📈 Prediction Quality:")
print(f"Standard deviation of residuals: {residual_std:.2f}")
print(f"95% of predictions within: ±{residual_std * 1.96:.2f} kg/ha")
print(f"Best prediction: error of {abs(residuals).min():.2f} kg/ha")
print(f"Worst prediction: error of {abs(residuals).max():.2f} kg/ha")

# Yield ranges
low_yield = y_test.quantile(0.25)
high_yield = y_test.quantile(0.75)

low_mask = y_test <= low_yield
high_mask = y_test >= high_yield

if low_mask.any():
	low_accuracy = r2_score(y_test[low_mask], y_pred_final[low_mask])
	print(f"R² for low yield farms (<{low_yield:.0f} kg/ha): {low_accuracy:.3f}")

if high_mask.any():
	high_accuracy = r2_score(y_test[high_mask], y_pred_final[high_mask])
	print(f"R² for high yield farms (>{high_yield:.0f} kg/ha): {high_accuracy:.3f}")


In [ ]:
# Save the cleaned and feature engineered dataset
if 'df_clean' in locals():
	# Add target variable back to the features
	final_dataset = pd.concat([X, y], axis=1)
	
	# Save to CSV
	final_dataset.to_csv('processed_dataset1.csv', index=False)
	print(f"Saved processed dataset with {final_dataset.shape[0]} rows and {final_dataset.shape[1]} columns")
	
	# Print key statistics
	print("\nKey features in saved dataset:")
	print(f"Numeric features: {len(numerical_cols)}")
	print(f"Categorical features: {len(categorical_cols)}")
	print(f"Target variable range: {y.min():.2f} - {y.max():.2f}")

In [ ]:
import numpy as np

import matplotlib.pyplot as plt

# Create the scatter plot
plt.figure(figsize=(10, 8))

# Plot scatter points
plt.scatter(y_test, y_pred_final, alpha=0.5, color='blue', label='Predictions')

# Add perfect prediction line (y=x)
min_val = min(min(y_test), min(y_pred_final))
max_val = max(max(y_test), max(y_pred_final))
plt.plot([min_val, max_val], [min_val, max_val], 'r--', label='Perfect Prediction')

# Add 10% error bands
plt.plot([min_val, max_val], [min_val*1.1, max_val*1.1], 'g--', alpha=0.3)
plt.plot([min_val, max_val], [min_val*0.9, max_val*0.9], 'g--', alpha=0.3)
plt.fill_between([min_val, max_val], 
				[min_val*0.9, max_val*0.9], 
				[min_val*1.1, max_val*1.1], 
				alpha=0.1, color='green',
				label='±10% Error Band')

# Add labels and title
plt.xlabel('Actual Yield (kg/ha)')
plt.ylabel('Predicted Yield (kg/ha)')
plt.title('Actual vs Predicted Rice Yield\n(R² = {:.3f})'.format(final_r2))

# Add grid
plt.grid(True, alpha=0.3)

# Add legend
plt.legend()

# Add text box with metrics
metrics_text = f'RMSE: {final_rmse:.0f} kg/ha\nMAE: {final_mae:.0f} kg/ha\nWithin 10%: {within_10_percent:.1f}%'
plt.text(0.05, 0.95, metrics_text, 
		 transform=plt.gca().transAxes,
		 bbox=dict(facecolor='white', alpha=0.8),
		 verticalalignment='top')

plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8')  # Visualization style

# === 1. Actual vs Predicted Plot ===
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred_final, alpha=0.5, color='blue')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', label='Perfect Prediction')
plt.xlabel('Actual Yield (kg/ha)')
plt.ylabel('Predicted Yield (kg/ha)')
plt.title(f'Model Predictions vs Actual Values\n(R² = {final_r2:.3f})')
plt.legend()
plt.show()

# === 2. Model Performance Comparison ===
plt.figure(figsize=(8, 6))
model_names = ['LightGBM', 'XGBoost', 'RandomForest', 'GradientBoost', 'ElasticNet', 'Ensemble']
r2_scores = [0.904, 0.904, 0.766, 0.855, 0.777, 0.900]
colors = ['#2ecc71', '#3498db', '#e74c3c', '#f1c40f', '#9b59b6', '#1abc9c']

plt.bar(model_names, r2_scores, color=colors)
plt.xticks(rotation=45)
plt.ylabel('R² Score')
plt.title('Model Performance Comparison')
plt.ylim(0.7, 1.0)
plt.show()

# === 3. Top 10 Feature Importance ===
plt.figure(figsize=(8, 6))
top_10_features = feature_importances[best_model_name].head(10)
plt.barh(np.arange(len(top_10_features)), top_10_features['importance'], color='#2ecc71')
plt.yticks(np.arange(len(top_10_features)), top_10_features['feature'])
plt.xlabel('Importance Score')
plt.title('Top 10 Most Important Features')
plt.gca().invert_yaxis()  # Make most important feature appear at top
plt.show()

# === 4. Prediction Error Distribution ===
plt.figure(figsize=(8, 6))
residuals = y_test - y_pred_final
sns.histplot(residuals, kde=True, color='#3498db')
plt.axvline(x=0, color='r', linestyle='--')
plt.xlabel('Prediction Error (kg/ha)')
plt.ylabel('Count')
plt.title(f'Prediction Error Distribution\nMean Error: {residuals.mean():.2f} kg/ha')
plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Assuming you already have results_df (your model results table as pandas DataFrame)
# Example: results_df = pd.DataFrame(your_results)

# === 1. R² Score Comparison ===
plt.figure(figsize=(8, 6))
sns.barplot(x="Model", y="R²", data=results_df, palette="viridis")
plt.xticks(rotation=45, ha="right")
plt.ylim(0.7, 1.0)
plt.ylabel("R² Score")
plt.title("Model Performance Comparison (R²)")
plt.show()

# === 2. RMSE Comparison ===
plt.figure(figsize=(8, 6))
sns.barplot(x="Model", y="RMSE", data=results_df, palette="magma")
plt.xticks(rotation=45, ha="right")
plt.ylabel("RMSE")
plt.title("Model Performance Comparison (RMSE)")
plt.show()

# === 3. MAE Comparison ===
plt.figure(figsize=(8, 6))
sns.barplot(x="Model", y="MAE", data=results_df, palette="coolwarm")
plt.xticks(rotation=45, ha="right")
plt.ylabel("MAE")
plt.title("Model Performance Comparison (MAE)")
plt.show()


In [ ]:
# Create a DataFrame with predictions and actual values
results_df = pd.DataFrame({
	'Actual_Yield': y_test,
	'Predicted_Yield': y_pred_final,
	'Absolute_Error': abs(y_test - y_pred_final),
	'Percentage_Error': abs((y_test - y_pred_final) / y_test) * 100
})

# Add prediction quality indicators
results_df['Within_10%'] = (results_df['Percentage_Error'] <= 10).astype(int)

# Save to CSV
results_df.to_csv('yield_predictions.csv', index=True)
print(f"Saved predictions and analysis to yield_predictions.csv")
print(f"Total predictions: {len(results_df)}")
print(f"Predictions within 10% accuracy: {results_df['Within_10%'].mean()*100:.2f}%")

In [ ]:
import seaborn as sns

import matplotlib.pyplot as plt

# Set style for better visualization
plt.style.use('seaborn-v0_8')

# Create a figure with 2x2 subplots
fig = plt.figure(figsize=(15, 12))

# 1. Model Performance Comparison (R² Scores)
plt.subplot(2, 2, 1)
model_performance = {
	'LightGBM': 0.904,
	'XGBoost': 0.904,
	'RandomForest': 0.766,
	'GradientBoost': 0.855,
	'Voting': 0.900
}
colors = ['#2ecc71', '#3498db', '#e74c3c', '#f1c40f', '#9b59b6']
plt.bar(model_performance.keys(), model_performance.values(), color=colors)
plt.title('Model Performance Comparison (R²)', fontsize=12)
plt.ylim(0.7, 1.0)
plt.xticks(rotation=45)
plt.ylabel('R² Score')

# 2. Top 10 Feature Importance
plt.subplot(2, 2, 2)
top_10_features = feature_importances[best_model_name].head(10)
plt.barh(range(len(top_10_features)), top_10_features['importance'], color='#2ecc71')
plt.yticks(range(len(top_10_features)), top_10_features['feature'])
plt.title('Top 10 Most Important Features', fontsize=12)
plt.xlabel('Importance Score')

# 3. Actual vs Predicted Plot
plt.subplot(2, 2, 3)
plt.scatter(y_test, y_pred_final, alpha=0.5, color='#3498db')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', label='Perfect Prediction')
plt.xlabel('Actual Yield (kg/ha)')
plt.ylabel('Predicted Yield (kg/ha)')
plt.title(f'Predictions vs Actual Values (R² = {final_r2:.3f})', fontsize=12)
plt.legend()

# 4. Error Distribution
plt.subplot(2, 2, 4)
residuals = y_test - y_pred_final
sns.histplot(residuals, kde=True, color='#3498db')
plt.axvline(x=0, color='r', linestyle='--')
plt.xlabel('Prediction Error (kg/ha)')
plt.ylabel('Count')
plt.title('Prediction Error Distribution', fontsize=12)

# Adjust layout
plt.tight_layout()

# Add main title
fig.suptitle('Finding the Optimal Rice Yield Prediction Model', fontsize=14, y=1.02)

plt.show()

In [ ]:
import seaborn as sns
import numpy as np

import matplotlib.pyplot as plt

# Set style for better visualization
plt.style.use('seaborn-v0_8')

# Create figure with subplots
fig = plt.figure(figsize=(20, 12))

# 1. Model Performance Comparison (R² and RMSE)
ax1 = plt.subplot(2, 2, 1)
model_metrics = pd.DataFrame({
	'Model': ['LightGBM', 'XGBoost', 'RandomForest', 'GradientBoost', 'Voting'],
	'R²': [0.904, 0.904, 0.766, 0.855, 0.900],
	'RMSE': [385.88, 386.20, 601.93, 474.85, 394.51]
})

x = np.arange(len(model_metrics['Model']))
width = 0.35

ax1.bar(x - width/2, model_metrics['R²'], width, label='R²', color='#2ecc71')
ax1_twin = ax1.twinx()
ax1_twin.bar(x + width/2, model_metrics['RMSE'], width, label='RMSE', color='#e74c3c')

ax1.set_ylabel('R² Score', color='#2ecc71')
ax1_twin.set_ylabel('RMSE', color='#e74c3c')
ax1.set_xticks(x)
ax1.set_xticklabels(model_metrics['Model'], rotation=45)
ax1.set_title('Model Performance Comparison')
ax1.grid(True, alpha=0.3)

# Add legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax1_twin.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

# 2. Top 10 Feature Importance
ax2 = plt.subplot(2, 2, 2)
top_10_features = important_features.head(10)
sns.barplot(x='importance', y='feature', data=top_10_features, 
			palette='viridis', ax=ax2)
ax2.set_title('Top 10 Most Important Features')
ax2.set_xlabel('Importance Score')
ax2.grid(True, alpha=0.3)

# 3. Actual vs Predicted Plot
ax3 = plt.subplot(2, 2, 3)
ax3.scatter(y_test, y_pred_final, alpha=0.5, color='#3498db')
ax3.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 
		 'r--', label='Perfect Prediction')

# Add 10% error bands
ax3.fill_between([y_test.min(), y_test.max()],
				[y_test.min()*0.9, y_test.max()*0.9],
				[y_test.min()*1.1, y_test.max()*1.1],
				alpha=0.2, color='green', label='±10% Error Band')

ax3.set_xlabel('Actual Yield (kg/ha)')
ax3.set_ylabel('Predicted Yield (kg/ha)')
ax3.set_title(f'Predictions vs Actual Values\n(R² = {final_r2:.3f})')
ax3.legend()
ax3.grid(True, alpha=0.3)

# 4. Error Distribution
ax4 = plt.subplot(2, 2, 4)
residuals = y_test - y_pred_final
sns.histplot(residuals, kde=True, color='#3498db', ax=ax4)
ax4.axvline(x=0, color='r', linestyle='--', label='Zero Error')
ax4.set_xlabel('Prediction Error (kg/ha)')
ax4.set_ylabel('Count')
ax4.set_title('Prediction Error Distribution\n'
			  f'RMSE: {final_rmse:.0f} kg/ha, Within 10%: {within_10_percent:.1f}%')
ax4.legend()
ax4.grid(True, alpha=0.3)

# Adjust layout and add main title
plt.tight_layout()
fig.suptitle('Comparing and Selecting the Optimal Model from Hyper-Optimized Ensemble', 
			 fontsize=16, y=1.02)

plt.show()

In [ ]:
# First, let's create a proper DataFrame for plotting
model_comparison = pd.DataFrame({
	'Model': ['LightGBM', 'XGBoost', 'RandomForest', 'GradientBoost'],
	'R²': [0.904, 0.904, 0.766, 0.855],
	'RMSE': [385.88, 386.20, 601.93, 474.85],
	'MAE': [156.52, 149.72, 388.23, 265.56]
})

# === 1. R² Score Comparison ===
plt.figure(figsize=(8, 6))
ax = sns.barplot(x='Model', y='R²', data=model_comparison, palette="viridis")
plt.xticks(rotation=45, ha="right")
plt.ylim(0.7, 1.0)
plt.ylabel("R² Score")
plt.title("Model Performance Comparison (R²)")

# Add numbers on top of bars
for i, v in enumerate(model_comparison['R²']):
	ax.text(i, v, f'{v:.3f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

# === 2. RMSE Comparison ===
plt.figure(figsize=(8, 6))
ax = sns.barplot(x='Model', y='RMSE', data=model_comparison, palette="magma")
plt.xticks(rotation=45, ha="right")
plt.ylabel("RMSE")
plt.title("Model Performance Comparison (RMSE)")

# Add numbers on top of bars
for i, v in enumerate(model_comparison['RMSE']):
	ax.text(i, v, f'{v:.1f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

# === 3. MAE Comparison ===
plt.figure(figsize=(8, 6))
ax = sns.barplot(x='Model', y='MAE', data=model_comparison, palette="coolwarm")
plt.xticks(rotation=45, ha="right")
plt.ylabel("MAE")
plt.title("Model Performance Comparison (MAE)")

# Add numbers on top of bars
for i, v in enumerate(model_comparison['MAE']):
	ax.text(i, v, f'{v:.1f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# === Example variables (replace with your actual data) ===
# y_test = ...
# y_pred_final = ...
# final_r2 = 0.9039
# final_rmse = 385.88
# within_10_percent = 86.4

# === Create 2x2 figure layout ===
fig = plt.figure(figsize=(20, 10))

# === 1️⃣ Actual vs Predicted Plot ===
plt.subplot(2, 2, 1)
plt.scatter(y_test, y_pred_final, alpha=0.5, color='#3498db')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 
         'r--', label='Perfect Prediction')

# ±10% error band
plt.fill_between([y_test.min(), y_test.max()],
                 [y_test.min() * 0.9, y_test.max() * 0.9],
                 [y_test.min() * 1.1, y_test.max() * 1.1],
                 alpha=0.2, color='green', label='±10% Error Band')

plt.xlabel('Actual Yield (kg/ha)')
plt.ylabel('Predicted Yield (kg/ha)')
plt.title(f'1️⃣ Predictions vs Actual Values\n(R² = {final_r2:.3f})')
plt.legend()
plt.grid(True, alpha=0.3)

# === 2️⃣ Error Distribution ===
plt.subplot(2, 2, 2)
residuals = y_test - y_pred_final
sns.histplot(residuals, kde=True, color='#3498db')
plt.axvline(x=0, color='r', linestyle='--', label='Zero Error')
plt.xlabel('Prediction Error (kg/ha)')
plt.ylabel('Count')
plt.title(f'2️⃣ Prediction Error Distribution\nRMSE: {final_rmse:.0f} kg/ha, Within 10%: {within_10_percent:.1f}%')
plt.legend()
plt.grid(True, alpha=0.3)

# === 3️⃣ Boxplot of Actual vs Predicted ===
plt.subplot(2, 2, 3)
plt.boxplot([y_test, y_pred_final], labels=['Actual', 'Predicted'])
plt.ylabel('Yield (kg/ha)')
plt.title('3️⃣ Distribution Comparison (Boxplot)')
plt.grid(True, alpha=0.3)

# === 4️⃣ Error vs Predicted Value ===
plt.subplot(2, 2, 4)
plt.scatter(y_pred_final, residuals, alpha=0.5, color='#3498db')
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel('Predicted Yield (kg/ha)')
plt.ylabel('Error (kg/ha)')
plt.title('4️⃣ Error vs Predicted Value')
plt.grid(True, alpha=0.3)

# === Adjust layout and add main title ===
plt.tight_layout()
fig.suptitle('🌾 Model Prediction Analysis Dashboard 🌾', 
             fontsize=18, y=1.03)

plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# === Example variables (replace with your actual data) ===
# y_test = ...
# y_pred_final = ...
# final_r2 = 0.9039
# final_rmse = 385.88
# within_10_percent = 86.4

# === Create 2x2 figure layout ===
fig = plt.figure(figsize=(20, 10))

# === 1️⃣ Actual vs Predicted Plot ===
plt.subplot(2, 2, 1)
plt.scatter(y_test, y_pred_final, alpha=0.5, color='#3498db')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 
         'r--', label='Perfect Prediction')

# ±10% error band
plt.fill_between([y_test.min(), y_test.max()],
                 [y_test.min() * 0.9, y_test.max() * 0.9],
                 [y_test.min() * 1.1, y_test.max() * 1.1],
                 alpha=0.2, color='green', label='±10% Error Band')

plt.xlabel('Actual Yield (kg/ha)')
plt.ylabel('Predicted Yield (kg/ha)')
plt.title(f'1️⃣ Predictions vs Actual Values\n(R² = {final_r2:.3f})')
plt.legend()
plt.grid(True, alpha=0.3)

# === 2️⃣ Error Distribution ===
plt.subplot(2, 2, 2)
residuals = y_test - y_pred_final
sns.histplot(residuals, kde=True, color='#3498db')
plt.axvline(x=0, color='r', linestyle='--', label='Zero Error')
plt.xlabel('Prediction Error (kg/ha)')
plt.ylabel('Count')
plt.title(f'2️⃣ Prediction Error Distribution\nRMSE: {final_rmse:.0f} kg/ha, Within 10%: {within_10_percent:.1f}%')
plt.legend()
plt.grid(True, alpha=0.3)

# === 3️⃣ Boxplot of Actual vs Predicted ===
plt.subplot(2, 2, 3)
plt.boxplot([y_test, y_pred_final], labels=['Actual', 'Predicted'])
plt.ylabel('Yield (kg/ha)')
plt.title('3️⃣ Distribution Comparison (Boxplot)')
plt.grid(True, alpha=0.3)

# === 4️⃣ Error vs Predicted Value ===
plt.subplot(2, 2, 4)
plt.scatter(y_pred_final, residuals, alpha=0.5, color='#3498db')
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel('Predicted Yield (kg/ha)')
plt.ylabel('Error (kg/ha)')
plt.title('4️⃣ Error vs Predicted Value')
plt.grid(True, alpha=0.3)

# === Adjust layout and add main title ===
plt.tight_layout()
fig.suptitle('🌾 Model Prediction Analysis Dashboard 🌾', 
             fontsize=18, y=1.03)

plt.show()


In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred_final, alpha=0.5, color='#3498db')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', label='Perfect Prediction')

# ±10% error band
plt.fill_between([y_test.min(), y_test.max()],
                 [y_test.min() * 0.9, y_test.max() * 0.9],
                 [y_test.min() * 1.1, y_test.max() * 1.1],
                 alpha=0.2, color='green', label='±10% Error Band')

plt.xlabel('Actual Yield (kg/ha)')
plt.ylabel('Predicted Yield (kg/ha)')
plt.title(f'Predictions vs Actual Values\n(R² = {final_r2:.3f})')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
residuals = y_test - y_pred_final

plt.figure(figsize=(8, 6))
sns.histplot(residuals, kde=True, color='#3498db')
plt.axvline(x=0, color='r', linestyle='--', label='Zero Error')
plt.xlabel('Prediction Error (kg/ha)')
plt.ylabel('Count')
plt.title(f'Prediction Error Distribution\nRMSE: {final_rmse:.0f} kg/ha, Within 10%: {within_10_percent:.1f}%')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Assuming residuals, final_rmse, and within_10_percent are already defined
# If not, compute mean_error = np.mean(residuals)
mean_error = np.mean(residuals)  # This should be approx. -16.07 kg/ha

plt.figure(figsize=(8, 6))
sns.histplot(residuals, kde=True, color='#3498db', bins=50)
plt.axvline(x=0, color='r', linestyle='--', label='Zero Error')
plt.axvline(x=mean_error, color='orange', linestyle='-', label=f'Mean Error ({mean_error:.0f} kg/ha)')

# Zoom x-axis to focus on central distribution
plt.xlim(-1500, 1500)

# Add annotation for 90% band
plt.axvspan(-1000, 1000, alpha=0.1, color='green', label='90% Errors Within ±1000 kg/ha')
plt.text(0.02, 0.98, '~90% errors ≤ ±1000 kg/ha', transform=plt.gca().transAxes, 
         va='top', ha='left', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.xlabel('Prediction Error (kg/ha)')
plt.ylabel('Count')
plt.title(f'Prediction Error Distribution\nRMSE: {final_rmse:.0f} kg/ha, Within 10%: {within_10_percent:.1f}%')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
residuals = y_test - y_pred_final

plt.figure(figsize=(8, 6))
plt.scatter(y_pred_final, residuals, alpha=0.5, color='#3498db')
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel('Predicted Yield (kg/ha)')
plt.ylabel('Error (kg/ha)')
plt.title('Error vs Predicted Value')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8, 6))
plt.boxplot([y_test, y_pred_final], labels=['Actual', 'Predicted'])
plt.ylabel('Yield (kg/ha)')
plt.title('Distribution Comparison (Boxplot)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
